## Navigating the command line

NOTE: I WROTE THIS SECTION THINKING IT WOULD BE ON CANNON; WILL HAVE TO REWORK (OR DELETE)

This workshop assumes you have a Cannon cluster account and some familiarity with Unix file systems, but let's do a quick review for how we orient ourselves and move around in a file system.

A file system is a hierarchical tree of directories (aka "folders"), with files stored within those directories. The "highest" level directory is the `root` directory (specified by `/`); all other directories are stored recursively within the root directory and are "lower" in the tree. We can check our position within the tree using the Unix command `pwd` (which stands for `p`resent `w`orking `d`irectory):

In [ ]:
pwd

This prints our current path, i.e. a list of all directories separated by a `/` starting from the root on down. You can think of a path as an "address" where something "lives" in a file system. When we log into the cluster (or when we open a new session on a local machine), we'll start in our default `home` directory. To display the contents of our current directory, we use the `ls` command:

In [ ]:
ls

This prints out a list of files in the directory, as well as any subdirectories, i.e. directories that are nested further down in the tree. As we did not specify any after our `ls` command, it listed the content of our current directory, but we can also give a path to `ls` and it will list the contents of the directory at that path, e.g.

In [ ]:
ls ph_dir

In this example we are specifying a *relative path*, meaning relative to where we currently are in the tree; in other words, the `ls` commands looks for a directory called `ph_dir` in our current directory. We can instead use an *absolute path* to a directory or file, which list the complete address starting from the root:

In [ ]:
ls /home/dkhost/ph_dir

PH:
Arguments to ls

Changing dirs with cd

Using find

Input/output

### Input and output
Unix commands have three default data "streams": standard input (`stdin`), which passes data into a program (such as `ls`, `pwd`, etc.) usually by the keyboard. Standard output (`stdout`) is where the output of a command is sent, default usually being the screen. The final stream, standard error (`stderr`) seperates output specifically for error messages, though by default it will also output to the screen.


### Visualizing files
There are numerous tools that are a part of most Unix-like systems that we can use to display the contents of a file:

-   `cat`: prints entire file to screen (short for conCATenate)
-   `head`: prints the first 10 lines of a file by default. We can change how many lines printed by adding the `-n #` argument, e.g. `head -n 5` prints only the first five lines
-   `tail`: the opposite of `head`, it prints the *last* 10 lines of a file
-   `less`: rather than printing the entire file contents to screen at once, `less` allows you to scroll through using the arrow keys (close `less` by pressing 'q'). This is the tools we will be using the most, as it is handy for large files that are typical of bioinformatics

## Sequence files 
### Intro to FASTA format
The first bioinformatic file type we'll be discussing is FASTA files, which is one of the most common formats used to represent biological sequences. FASTA as a format was originally taken from a bioinformatic software package written back in the 1980s, and stuck around and became a standard. A FASTA file is composed of a number of *entries*, which can be either protein or nucleotide sequences. An "entry" can represent a number of things, e.g.:

- Chromosomes in a genome assembly
- mRNA transcript sequences
- Translated protein sequences
- Etc.

Importantly, while the content of FASTA files can vary, the syntax is the same. Each entry is comprised of two lines:
- *Header line*, which starts with a `>` symbol. This contains the unique name for each entry (e.g. >chr1), optionally followed by additional metadata.
- *Sequence line*, which follows a header line. This is a string of either nucleotides or amino acids and represents the actual sequence

**Exercise**: open an example FASTA file at the command line; print first couple entries

The order is important, as the sequence must follow its corresponding header. However, for some FASTA files, the sequence line can be set to wrap every certain number of characters (e.g. every 50 nucleotides). These are referred to as *multi-line FASTA*, and can make your life difficult...make sure you know whether your FASTA is multi-line or single-line! For example,

```text
>sequence1
ATGGACGCTAGTCAGTAGATGCATGCTGACCCAACATAACG
```

vs.

```text
>sequence1
ATGGACGCTAG
TCAGTAGATGC
ATGCTGACCCA
ACATAACG
```

The sequences in these two examples are identical, but one is broken up by lines. Most programs can automatically handle both types of **FASTA** files, but as we stated above, if you parse your own **FASTA** files, you need to be aware of the difference! For this workshop however, we will usually be dealing with single-line files.

### Manipulating FASTA files
We've viewed the contents of FASTA files, now let's begin to learn how we can starting pulling useful information out of them! One of the most basic things you often want to know about your FASTA file is how many sequences are in it; for example, say you downloaded a genome assembly off NCBI and want to know how many scaffolds it has. A command line tool we can use for this is `wc` (which stands for `w`ord `c`ount). As the name suggests, `wc` will count the nuymber of words in a file. This isn't super useful for what we're looking for, however we can modify `wc`'s behavior by providing it with the argument `-l`, which will cause `wc` to instead count the number of *lines* in the file:

In [ ]:
wc -l ph.fasta

This is close to what we want, but remember that each entry in a FASTA file is comprised of two lines (a header and a sequence line), and we want to know just the number of entries in our FASTA. We could just divide the number of lines by two, but that is kind of clunky for large files (and also would not work if our FASTA is multi-line!). Instead, let's introduce one of the most powerful tools in our bioinformatics toolkit.

### Searching with `grep`
`grep` is a powerful command-line search tools that is included as part of Unix-like systems. At the most basic level, `grep` searches for a string of characters that match a pattern and will print lines containing a match. The basic syntax is as follow:

In [ ]:
grep 'pattern' file_to_search.txt

This may seem simple, but `grep` is one of the most useful tools in bioinformatics! So for our problem, we know that each entry has a header and a sequence line...

**Exercise**: write a grep command that matches the headers in the FASTA file

This is close to what we want, but we need to count the header lines as well. To do this we need to use a **pipe**, denoted by the character `|`, which is a core part of Unix and is the backbone of bioinformatics pipelines. Normally when we run a command, the output of the command is printed directly on the screen (i.e. standard output or `stdout`). A pipe instead directs the output from one command as *input* to another command! So if we use `grep` to match and print header lines, we can pipe that directly to `wc -l` to count the number of entries:

In [ ]:
grep '>' ph.fasta | wc -l

You can use pipes to string together entire chains of commands! Certain programs may differ in how they accept input or direct output, which can affect how pipes behave, but we will discuss this more when we get to more specialized command line tools.


By default, `grep` will return a match if *any part of the string* matches your pattern. For instance, say we wanted to pull out the headers that correspond only to **chr**omosomes. If you attempt to match pattern `c` what would happen?

> **Exercise**:
> In the code block below, write a grep command to print all lines that contain the 'c' character in the FASTA file:

In [ ]:
## Write a command to display all lines with the 'c' character
# data/test.fa
# !grep 'c' data/test.fa
## Write a command to display all lines with the 'c' character

In [ ]:
#@title Solution {display-mode: "form"}
grep 'c' data/test.fa

You can see that not only are we pulling out headers that do not correspond to chromosomes, we are even getting a sequence line that contains a lowercase 'c'! We would instead need be more specific with the string we are trying to search for.

> Run the code block below to print all the lines that contain the '>chr' string in the FASTA file:

In [ ]:
grep '>chr' data/test.fa
# grep: The Unix string search command
# '>chr': The string to search for in the provided file

This is getting better...notice that by matching `>chr` we are correctly getting the line '>chromosome4', as it is still a partial match. However, we are still missing a sequence, '>Chr3'! This is because by default `grep` is *case-sensitive*. Thankfully, we can fix that.

### Modifying `grep` with arguments
Part of what makes `grep` so powerful is that it take a huge number of arguments that modify how it behaves and allows it to match much more advanced patterns than a simple literal match. We can't cover all of them, but we'll highlight a few that are especially useful.

`grep -i` allows case-insensitive matches. So to return to our above problem, we can specify to ignore the case.

> Run the code block below to print all lines that contain the '>chr' string in the FASTA file, ignoring the case of the letters in the string:

In [ ]:
grep -i '>chr' data/test.fa
# grep: The Unix string search command
# -i: An option the tells grep to ignore the case of the matches, e.g. >chr will match >CHr and >Chr, etc., as well as >chr
# '>chr': The string to search for in the provided file

`grep -c` counts the number of times a match occurs. One of the most useful applications of this is to determine how many entries there are in a FASTA file.

> Run the code block below to use grep to count the number of sequences in a FASTA file:

In [ ]:
grep -c '>' data/test.fa
# grep: The Unix string search command
# -c: An option the tells grep to simply count the number of lines that contain the provided string
# '>': The string to search for in the provided file

`grep -v` *inverts* grep, printing every line that does NOT match the pattern. E.g. we want to pull out just the sequences from a FASTA file and not the headers.

> **Exercise**:
> In the code block below, write a command to use grep to print out only the lines that contain sequence and not the headers in the FASTA file:

In [ ]:
## Use grep to display only sequence lines (EXCLUDE header lines)
# data/test.fa
# !grep -v '>' data/test.fa
## Use grep to display only sequence lines (EXCLUDE header lines)

In [ ]:
#@title Solution {display-mode: "form"}
grep -v '>' data/test.fa

There are also several options that display not only the line that contains the matching string, but the lines before and/or after it:

- `grep -A [n]` returns matching line and n lines *after* match\
- `grep -B [n]` returns matching line and n lines *before* match\
- `grep -C [n]` returns matching line and n lines *before and after* match

We can use `grep -A` to pull out both the header and the sequence for a particular entry of interest (assuming that the FASTA file is single-line and not multi-line!).

> Run the code block below to print both the headers that contain a certain string as well as the sequences (since this is not a multi-line FASTA file):

In [ ]:
grep -A 1 '>chr1' data/test.fa
# grep: The Unix string search command
# -A 1: An option the tells grep to display the line right after each line that contains the provided string as well as the line with the match
# '>chr1': The string to search for in the provided file

Note that this is actually pulling out **two** entries, due to the partial matching of the pattern we used. To get around this problem, we can use `grep -w`, which forces `grep` to match *entire words*, in combination with the -A argument.

> Run the code block below to print both the headers that contain an exact match of a certain string as well as the sequences (since this is not a multi-line FASTA file):

In [ ]:
grep -A 1 -w '>chr1' data/test.fa
# grep: The Unix string search command
# -A 1: An option the tells grep to display the line right after each line that contains the provided string as well as the line with the match
# -w: This option tells grep to only print lines that EXACTLY match the provided string
# '>chr1': The string to search for in the provided file

> **Exercise**:
> In the code block below, write a grep command that searches for a particular sequence motif in our FASTA file and prints the whole line containing that sequence as well as the sequence header associated with that sequence. Remember that the FASTA sequence is not multi-line.
> Search for the following sequence motif: GGGTCGTCGT

In [ ]:
## Write a grep command to search for a sequence motif and display the matched sequence and the header
# data/test.fa
# !grep -B 1 'GGGTCGTCGT' data/test.fa
## Write a grep command to search for a sequence motif and display the matched sequence and the header

In [ ]:
#@title Solution {display-mode: "form"}
grep -B 1 'GGGTCGTCGT' data/test.fa

#### Regular expressions

One last way that we can modify how `grep` behaves is with *regular expressions*, a.k.a *regex*. Regex are patterns that describe a *set of strings*. In other words, they allow you to match complex patterns with `grep` (and other Unix commands), not just exact matches! Regex syntax can be very convoluted, and if you are trying to match a complex pattern it is easiest to ask a LLM or the internet for the correct regex. However, there are a few basic regex patterns that are worth remembering, as they are useful in this sort of quick data wrangling:

-   `^` matches pattern at start of string
-   `$` matches pattern at end of string
-   `[ ]` matches any of enclosed characters. Can use in conjunction with 'A-Z' or '0-9', i.e.:
    -   `[A-Z|a-z]` matches any alphabetical character, upper or lower case
    -   `[0-9]` matches any numeric character

So a more careful way to count the number of entries in a FASTA file would be by matching all the lines that start with a `>` character.

## Modifying FASTA files
If you are doing more in-depth/complex modifications to a FASTA file you'll probably want to write a custom script to do it; however, for quick edits or cleanup, simple command line tools are still very useful. Let's take a look at a few of them.

### Editing strings with `sed`
The Unix utility `sed` (`s`tream `ed`itor) is one such tool that allows us to perform text transformations, either on files or as input from a pipe. It reads input line by line and performs some function on each line. A `sed` command is structured like this:

```
sed [ARGUMENTS] '[COMMAND]' [INPUT]
```

If an input file is not specified, `sed` assumes the input is piped from standard input (aka `STDIN`). One of the most useful commands is the substitute function (`s/`), which replaces the first matching string in a line with another string:

```
sed 's/string_to_replace/string_replacing/' [INPUT]
```

For example:

In [ ]:
echo "oh time time thy pyramids" | sed 's/time/test/'

Note that by default, `sed` only replaces the first instance of a string in a line...if we want it to replace ALL matching strings in a line, we append `g` (for `g`lobal) to the command:

In [ ]:
echo "oh time time thy pyramids" | sed 's/time/test/g'

> **Exercise**
> Write a command that replaces 'D.mel' with 'DMEL' in the FASTA file

In [ ]:
##Command here

In [ ]:
#@title Solution {display-mode: "form"}
sed 's/D.mel/DMEL/g' ph.fasta

> **Exercise**
> Write a command that outputs the sequence names of a FASTA file, WITHOUT the '>' character

In [ ]:
##Command here

In [ ]:
#@title Solution {display-mode: "form"}
grep '>' ph.fasta | sed s/>//'

`sed` outputs to STDOUT, which as usual we can direct to a file rather than printing to the screen. However we can also use `sed` to modify the input file *directly*, i.e. modify the file "in-place" using the `-i` argument. This is handy if we don't want to clog up our directory with duplicate or temporary files, but be very careful using this option as it does overwrite the original file!

Pro tip: I like to first run my `sed` command piped to `less` or `head`, just to double check that the command is doing what I think it is doing, before then running `sed` in-place:

In [ ]:
sed 's/D.mel/DMEL/g' ph.fasta | head

In [ ]:
#Looks good!
sed -i 's/D.mel/DMEL/g' ph.fasta

#### Other `sed` functions
While string replacement is probably the most common use of `sed`, it has lots of other functionality that lets us quickly pull useful info out of files, such as using `sed -n` to print *specific numbered lines of a file*. For example, if we wanted to print just the 6th line of a file: 

In [ ]:
sed -n '6p' ph.fasta

#the -n tells `sed` to not automatically print each line unless explicitly told to
#the 'p' command tells `sed` to print the line if it matches the pattern (in this case, line 6)

We can specify a range of lines to print, such as the first command which will print lines 5 thru 10 of the file, or multiple non-sequential lines using the `-e` argument to add multiple expressions, as in the second command which prints only lines 5 and 10:

In [ ]:
sed -n '5,10p' ph.fasta

In [ ]:
sed -n -e '5p' -e '10p' ph.fasta

Another useful thing we can do with `sed` is **delete** specific lines of a file! The syntax is similar to printing (though without the `-n` argument), as shown in this command which will delete the 6th line from the file:

In [ ]:
sed '6d' ph.fasta

#the 'd' command tells `sed` to delete the line if it matches the pattern (in this case, line 6)

In [ ]:
sed '6,7d' ph.fasta

In the above examples we referred to specific numbered lines as the pattern to match, but we can also provide a *specific string pattern* (similar to how we were substituting with `s/`) and delete any lines matching that pattern:

`sed '/pattern to match/d' [INPUT]`

So the following command would delete every line that matches a '>' character in our FASTA file:

In [ ]:
sed '/>/d' ph.fasta

Something to note: if you remember back to the `grep` section, you may realize that the above `sed` command is basically equivalent to running inverted `grep` (i.e. `grep -v '>'`), which returns all lines that do NOT contain a specified pattern. This is a fairly common situation, where different tools can fulfill similar functions and there are several different viable ways to accomplish your task (similar to how `grep | wc -l` and `grep -c` do the same thing). So, which do you use? In most cases, it is up to you! There might be some differences in efficiency (which could start to matter if your files are very large), or some considerations depending on what exactly you are trying to do. 

For example, if I am trying to directly modify a file in-place, I personally would prefer to use `sed` as it can be done in a single command using `-i`. I could use `grep` to direct output to a temporary file, then re-name that file:

```
grep -v '>' ph.fasta > temp
mv temp ph.fasta
```

But that is slightly more annoying to do :)

NOTE: IDK IF WE WANT THIS SECTION...IT MIGHT RAISE SOME TRICKY STUFF RE: "why can't you just pipe to mv" WHICH WOULD BE TOO MUCH

### Editting FASTA files using `tr`

NOTE: THIS MIGHT BE BETTER IN DAY 3 WHEN WE TALK ABOUT TABULAR DATA FOR THE "SQUEEZING" PART

One more basic command line utility that can be helpful when wrangling files is the `tr` tool, which `tr`anslates a text character into a different character or deletes certain characters. 


At first glance, `tr` seems like it does the exact same thing as `sed`, as it is used to replace text with other text, and in some cases they are interchangeable. However, there are some subtle differences that make `tr` a better tool for certain tasks.

The main difference is that `sed` works on strings and whole lines, `tr` processes single characters. So if you are replacing entire words you would want to use `sed`, while if you had some transformation on individual characters `tr` would be a better fit.

## Practical problems: what's wrong with my FASTA file??
To get even more comfortable with FASTA format as well as the command line tools we've discussed, we are going to try practicing how you might being to troubleshoot a pipeline involving a FASTA file. In this scenario, imagine that you received a genome assembly file in FASTA format from a colleague, and when you tried to input it into some kind of analysis program, it throws an error saying that the file does not match FASTA specifications.

**Discussion**: what are the steps we can do to diagnose this problem?

<details><summary>Solution</summary>

Some ways we could start troubleshooting:
- Open the file and look at it with `less`
- Check the first and last entries with `head` and `tail`
  
What if the file is very large, or the problem isn't immediately visible?
- Count the number of header lines vs number of sequence lines
- We can see that there are a mismatched number of header vs sequence lines, which suggests we have some malformatted headers
  
</details>

We've narrowed down our issue, now how do we actually identify the problematic sequence headers?

> **Exercise**
> Now that we figured out what the problem is, fix the file to make it proper FASTA syntax

In [ ]:
## Command here

In [ ]:
#@title Solution {display-mode: "form"}
sed -i 's/@/>/g' ph.fasta

## Seqkit
So far, we have been discussing generic text wrangler command line utilities that are part of most Unix systems and are useful for manipulating FASTA files. However, while they are powerful they were not designed specifically with FASTA or other sequence files in mind, so certain tasks might be difficult to accomplish using generic tools alone. Let's introduce our first specialized program `seqkit`, which is a [suite of tools](https://bioinf.shenwei.me/seqkit/) that were specifically designed to manipulate sequence files! We'll see that many of the tasks we have been using command line utilities for can also be done using `seqkit`, plus some extended functionality as well.

The downside to using `seqkit` vs generic tools is that it is not included by default as part of a Unix operating system like the other tools are, and must be downloaded manually. For this workshop we've already done that for you, but if you want to use it on your own you'll have to install it yourself. There are also plenty of alternative tools that have been developed as well, but we are focusing on `seqkit` as it is very functional, lightweight, and is actively maintained!

If we look at the [manual page](https://bioinf.shenwei.me/seqkit/usage/) for `seqkit`, we can see that it has a ton of different functions, with documentation and example useage for each. We aren't interested in all of these functions, so let's focus on a few common tasks.

### Manipulating and summarizing FASTA files
One very useful way to sanity check our FASTA files is checking sequence lengths; we can use `seqkit stats` to get a quick summary of length distribution, as well as some other basic summary statistics: 

In [ ]:
seqkit stats ph.fasta

`seqkit seq` can transform sequences in a variety of ways depending on the arguments used, such as:

In [ ]:
#Convert lowercase to uppercase
seqkit seq -u ph.fasta

#Reverse complement sequence
seqkit seq -r -p ph.fasta

#Filter sequences shorter than a certain length (e.g., 100 bp)
seqkit seq -m 100 ph.fasta

Like the generic command line tools, we can use pipes to direct the input and output of `seqkit` commands into each other, allowing for some more advanced processing. For example, as detailed in the manual page for `seqkit stats`, the `stats` module will count any gaps that are present in the sequence, so if we wanted to get a stats summary of the file that does NOT include any gaps, we could use `seqkit seq -g` to remove gaps, then pipe to `seqkit stats` to get our summary:

In [ ]:
seqkit seq -g ph.fasta | seqkit stats

Let's also look at `seqkit grep`, which has a familiar name! As the name suggests, it is inspired by command line `grep`, performing a similar function and shares a similar syntax. However, as `seqkit` was designed with FASTA files in mind, it has some additional handy functionality compared to normal `grep`. For example, we can search for a particular ID in the header, and it will return both the matching header AND the associated sequence: 


In [ ]:
seqkit grep chr1 ph.fasta

The default behavior is to match the entire sequence header but it can also do partial matching, including multiple patterns at once (and regex!): 

In [ ]:
#This will match any sequence that has 'chr' in the header
seqkit grep -p 'chr' ph.fasta

### Editting FASTA files
We discussed some basic editting using `sed`, but `seqkit replace` also fills a similar function, and allows for some more advanced editting. In basic useage, provide a pattern to match with `-p` and a replacement for the pattern with `-r`, e.g. if we wanted to change 'chr' to 'chromosome':

In [ ]:
seqkit replace -p 'chr' -r 'chromosome' ph.fasta

This is something you could also do with `sed`, however we can see in the example useage section of the documentation that we can see that we can make edits that would be much trickier to do if you were using just `sed`. For example, if we wanted to add a prefix string to each entry in the FASTA file, we can use the regex for "start of" (`^`) to prepend the string:

In [ ]:
seqkit replace -p ^ -r sequence_ ph.fasta